

### Primer trabajo práctico integrador


#### Introducción

En este trabajo se utilizo el modelo de Regresión Lineal, que plasma graficamente la relación entre una variable escalar dependiente “y” y variables explicativas “X”. 

El trabajo consta de predecir el valor de precio_prom de alquiler de un departamento en base a el año, barrio, comuna, trimestre, ambientes y estado en base a los datos recopilados.

Variables:
- barrio: zona geográfica donde se encuentra la propiedad
- año: año de observación
- trimestre: trimestre del año
- precio_prom: precio promedio del m² (USD)
- ambientes: cantidad de ambientes
- estado: condición del departamento (Usado o A estrenar)
- comuna: número de comuna de la propiedad

El objetivo de la regresión es producir un modelo que represente el "mejor ajuste" a los datos observados. Normalmente, el modelo es una función que describe algún tipo de curva, que está determinada por un conjunto de parámetros (por ejemplo, pendiente e intersección).

"Mejor ajuste" significa que hay un conjunto óptimo de parámetros de acuerdo con un criterio de evaluación que elegimos.

El modelo de regresión intenta predecir el valor de una variable, conocida como variable dependiente, variable de respuesta o etiqueta, utilizando los valores de otras variables, conocidas como variables independientes, variables explicativas o características.

En forma matemática, el objetivo de la regresión es encontrar una función de algunas características $X$ que predice el valor de la etiqueta $Y$

__¿cuáles son los mejores valores de  $a$ y $b$?__ En regresión lineal, $a$ y $b$ se eligen para minimizar el error cuadrado entre las predicciones y las etiquetas conocidas. Esta cantidad se conoce como SSE (Sum Squared Error), o sea la suma de las diferencias al cuadrado entre cada observación.  Para $n$ casos, la SSE se calcula de la siguiente manera:

$$SSE = \sum_{i=1}^n \big( f(x_i) - y_i \big)^2\\
= \sum_{i=1}^n \big( \hat{y}_i - y_i \big)^2\\
= \sum_{i=1}^n \big( a \cdot x_i + b - y_i \big)^2$$

[Más información](https://es.wikipedia.org/wiki/Suma_residual_de_cuadrados)

El enfoque de regresión que minimiza la SSE se conoce como el **método de mínimos cuadrados**.

La validación, optimización y regularización son procesos clave en el desarrollo de un modelo de regresión lineal. La validación verifica la bondad del modelo y su capacidad de generalización, la optimización busca los parámetros que mejor ajustan los datos, y la regularización evita el sobreajuste, mejorando la robustez del modelo.



##### Importación de los datos y carga del dataset 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("precio-venta-deptos.csv",sep=';')

print("🔹 Dimensiones:", df.shape) 

print(f"\n{df.head()}\n")

Observaciones:

- Se muestra en consola el tamaño del dataset. El cual es: 7298 filas x 7 columnas.
- Con df.head() se muestran por default las primeras 5 filas del dataset. Se percibe precio_prom como NaN
  en algunas filas.

Revision de datos:

In [ ]:

print(f"\n{df.info()}\n")

#Datos que faltan segun columna:
print(f"\n valores nulos:\n{df.isnull().sum()}\n")

Observaciones:

- En consola figura info general con los tipos de datos para cada columna.
- Se muestran por consola la cantidad de valores nulos en el dataframe por columna. Se perciben  4211
  valores nulos para la columna precio_prom y para las restantes columnas no se presentan valores nulos.

##### Despejamos los valores nulos del dataframe:

In [ ]:
df['precio_prom'] = pd.to_numeric(df['precio_prom'], errors='coerce')
df['año'] = pd.to_numeric(df['año'], errors='coerce')
df['trimestre'] = pd.to_numeric(df['trimestre'], errors='coerce')
df = df.dropna()

# confirmamos estructura final
print("\n nuestro dataset limpio:", df.shape)

Observaciones:

- Limpieza basica, se eliminan todas las columnas que posean valores nulos con dropna()
- Se muestra el tamaño del dataframe sin las filas con valores nulos. Tamaño = (3085,7)

##### Manejo de los outliers (valores atípicos)

In [ ]:
for col in df.select_dtypes(include=np.number).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # Reemplazamos los valores atípicos por la mediana de la columna
    median_value = df[col].median()
    df[col] = np.where((df[col] < lower) | (df[col] > upper), median_value, df[col])
    print("\nReemplazamos outliers por la mediana en las variables numéricas.")

Observaciones:

- Recorremos cada columna y mide el rango de dispersión de los datos con los quantiles.
- Compara cada columna con el lower y el upper permitidos y los que no cumplan se consideran outliers,
  sus valores son reemplazados por el calculo de la mediana de esa columna específica.

##### Evaluacion del sesgo de distribución de los valores sobre el eje x:

In [ ]:
print(f"\nLas medidas de asimetría son:\n{df.skew(numeric_only=True)}\n")

Observaciones:

- Las columnas año, trimestre, precio_prom y comuna tienen valores cercanos a 0, indican simetría y
  correspondencia con los datos. 
  
- Año:        -0.409707 tiene valor negativo indica sesgo hacia la izquierda
- Trimestre:   0.079995 tiene valor positivo indica sesgo hacia la derecha
- Precio_prom: 0.306623 tiene valor positivo indica sesgo hacia la derecha
- Comuna:     -0.283900 tiene valor negativo indica sesgo hacia la izquierda

##### Gráfico de distribución de asimetría para cada columna:

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns
skewness_values = df[numerical_cols].skew()

fig, axes = plt.subplots(nrows=1, ncols=len(numerical_cols), figsize=(4 * len(numerical_cols), 5))

if len(numerical_cols) == 1:
    axes = [axes]

for i, col in enumerate(numerical_cols):
    sns.histplot(data=df, x=col, ax=axes[i], kde=True)
    axes[i].set_title(f'{col}\nAsimetría: {skewness_values[col]:.2f}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

Observaciones:

- Las columnas año, trimestre, precio_prom y comuna tienen valores cercanos a 0, indican una cercanía a   simetría y correspondencia con los datos. 
  
- Año: En el gráfico se presenta mayor presencia de registro de datos en el año 2016, seguidos por el 2014 y el 2018 con frecuencias por encima de las 400.

- Trimestre: El primer trimestre tuvo la mayor taza de datos, seguido por el segundo trimestre por una leve cercanía, luego decae levemente el tercer y cuarto trimestre. Los datos presentan una buena distribución.

- Precio_prom: La mayoría de precios se encuentran dentro de la gama media-baja. Se percibe un incremento de tasa de datos aproximadamente a los 2300 USD con una frecuencia por encima de las 300.

- Comuna: La mayor tasa de datos de ambientes se presentan en comunas del norte y centricas. La comuna 12 presenta una frecuencia rozando los 350 seguida por comuna 1, comuna 11 y 13 con datos por encima de los 250.

##### Medidas de Kurtosis (muestra de datos alrededor de la media)

In [ ]:
print(f"Las medidas de kurtosis son:\n{df.kurt(numeric_only=True)}\n")

Observaciones:

- Año: -1.127803 Medida de kurtosis con un coeficiente -1, es una ligera desviación de la normalidad.

- Trimestre: -1.346515 Medida de kurtosis con un coeficiente -1, es una ligera desviación de la normalidad.

- Precio_prom: 0.038160 Medida de kurtosis con una valor normal alrededor de la media.

- Comuna:  -1.205979  Medida de kurtosis con un coeficiente -1, es una ligera desviación de la normalidad.

##### Distribución del precio promedio:

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df['precio_prom'], bins=30, kde=True)
plt.title('distribución del precio promedio del m²')
plt.show()

Observaciones:

- Se logra percibir una concentración promedio de departamentos con precios desde 1500 USD hasta los 3000 USD inclusive. La mayor concentración de departamentos es a los aproximadamente 2200 a 2300 USD alcanzando un recuento de 350 registros.

##### Boxplot por estado

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='estado', y='precio_prom', data=df)
plt.title('precio promedio según estado')
plt.show()

Observaciones:

- Usado: En el gráfico se percibe que los valores de costo promedio en este estado rondan desde por encima de los 1500 USD hasta casi los 2500 USD. 

- A estrenar: En el gráfico se percibe que los valores de costo promedio en este estado rondan desde los 2000 USD hasta por encima de los 2500 USD. 

##### Promedio por barrio:

In [ ]:
top_barrio = df.groupby('barrio')['precio_prom'].mean().sort_values(ascending=False).head(10)
top_barrio.plot(kind='bar', color='teal', figsize=(10,5))
plt.title('top 10 barrios con mayor precio promedio del m²')
plt.ylabel('precio promedio (USD/m²)')
plt.show()

Observaciones:

- Las zonas con departamentos con valores más elevados que superan los 2500 USD son: Colegiales; como primer lugar, Nuñez; segundo lugar, Retiro; tercero, Palermo; cuarto, Coghlan; quinto, Recoleta; sexto y Belgrano; como séptimo lugar.

- Luego Urquiza, Saavedra y Parque Chas superan valores de 2000 USD casi rozando los 2500 USD.

##### Optimización y testeo:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# codificar variables categoricas
df_encoded = pd.get_dummies(df, columns=['barrio', 'ambientes', 'estado', 'comuna'], drop_first=True)

# definir variables predictoras (X) y objetivo (y)
X = df_encoded.drop('precio_prom', axis=1)
y = df_encoded['precio_prom']

# division train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Observaciones:

- Elegimos las columnas categoricas como dummies o ficticias para el entrenamiento del modelo, _X_ va a ser el valor para entrenar por el modelo (barrio,año,comuna,ambientes,estado,trimestre).
- _y_ va a ser para los valores a objetivo a predecir.

Division train/test:
- 80% para entrenar los datos usando _X_train_ e _y_train_
- 20% para testear el desempeño del modelo _X_test_ e _y_test_ 


##### Escalado:

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("datos preparados para modelado.\n")

Observaciones:

- Se obtiene la mediana y la desviación estandar de las columnas en _X_train_ para lograr estandarizar los datos de las columnas por sustraer la mediana y dividir por la desviación estandar. Se asegura que los valores de la mediana estén entre 0 y una desviación estandar de 1 para asegurar la performance del modelo.

- Aplica en _X_test_ la estandarización de los datos de prueba utilizando ls parametros de media y desviación estandar a partir del conjunto de entrenamiento.


In [ ]:
from sklearn.linear_model import LinearRegression

#####  Regresor para el modelo

In [ ]:
regression = LinearRegression()
regression.fit(X_train, y_train)

from sklearn.metrics import mean_squared_error, r2_score

#### Objervaciones:
- El modelo utiliza _X_train_ e _y_train_ para aprender la relación de la variable independiente _X_ y la dependiente _y_ a partir de los datos de entrenamiento proporcionados. Encuentra la pendiente y el término independiente que minimizan el error cuadrático entre los valores reales del conjunto y los predichos.

In [ ]:
y_pred = regression.predict(X_test)
y_pred

##### Observaciones:
- Al aplicar .predict(), pasa el regresor como argumento y obtiene la respuesta predicha correspondiente.


##### Predicciones sobre el conjunto test:

In [ ]:
y_pred = regression.predict(X_test)

# Mostrar primeras predicciones
print(f"Primeras predicciones:\n{y_pred[:5]}\n")

##### Observaciones:
- Valores predichos de precio_prom para los registros _X_test_:

- 2142.38554758 ≈ 2142.38 USD/M² 
- 2852.60827576 ≈ 2852.60 USD/M² 
- 2221.7270787  ≈ 2221.72 USD/M² 
- 2869.10775233 ≈ 2869.10 USD/M² 
- 3044.042268   ≈ 3044.04 USD/M² 

##### Comprobar con valores reales:

In [ ]:
df_aux = pd.DataFrame({'Actual': y_test.values, 'Predicción': y_pred})
print("\nComparación de valores reales vs predichos:")
print(df_aux.head())

##### Observaciones:
- El modelo predice valores muy cercanos a los reales. Por ejemplo, en la primera fila el valor real era 2161 USD y el modelo predijo 2142 USD (una diferencia de solo 19 USD/m²).

- En otras observaciones hay más diferencia, pero el patrón general se mantiene estable y realista. Esto indica que el modelo se está ajustando correctamente a las relaciones entre las variables predictoras y la mediana de precio_prom.

#####  Evaluación del modelo por MSE Y R²:

In [ ]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nError cuadrático medio (MSE): {mse:.2f}\n")
print(f"Coeficiente de determinación R²: {r2:.4f}\n")

##### Observaciones:
- EL MSE calcula la diferencia entre los valores predichos y los valores reales, y luego se calcula el promedio de estas diferencias al cuadrado.

- El R² mide la proporción de la variabilidad en la variable dependiente que es explicada por la variable independiente.

- Error cuadrático medio (MSE): 54807.12

Para llegar a una conclusión con el MSE sacamos su raiz  ≈ 234.2 lo cual quiere decir que tiene una equivocación de ese valor, lo cual e sun error relativamente bajo, ya que hay valores que se presentan desde los 1000 USD hasta por encima de los 2500 USD. 

- Coeficiente de determinación R²: 0.8020

El modelo explica el 80.2 % de la variabilidad del precio promedio por M² a partir de las variables independientes (barrio, año, trimestre, ambientes, estado, comuna).

##### Escenario con un un departamento a predecir:

In [ ]:
nuevo_depto = pd.DataFrame({
    'barrio': ['Agronomia'], #Del barrio agronomia
    'año': [2011],            #en el año 2011
    'trimestre': [3],         #3er trimestre
    'ambientes': [2],         #"2 ambientes"
    'estado': ['Usado'],       #estado usado
    'comuna': [15],            #de la comuna 15

})

In [ ]:
#Asigno columnas del modelo al nuevo ejemplo
nuevo_depto = nuevo_depto.reindex(columns=X.columns, fill_value=0)

# Escalar con el mismo scaler usado en entrenamiento
nuevo_depto_scaled = scaler.transform(nuevo_depto)

# Predecir el precio
prediccion_nueva = regression.predict(nuevo_depto_scaled)
print(f"\nPredicción del precio promedio (USD/m²) para el nuevo departamento: {prediccion_nueva[0]:.2f}\n")

##### Observaciones:
- Un departamento usado de 2 ambientes, ubicado en Agronomía, registrado en el tercer trimestre de 2011, y perteneciente a la comuna 15, tendría un precio promedio estimado de aproximadamente 1955 USD/M².

- Está dentro del rango esperado según la distribución de precios entre 1500 y 3000 USD/M², lo que muestra que el modelo generaliza bien incluso para un caso nuevo.

In [ ]:
#Obtener b_0 y 𝑏_1.
print(f"Intercepción del modelo: {regression.intercept_}\n")
print(f"Coeficientes (regression.coef_), longitud: {len(regression.coef_)}\n")

##### Observaciones:

- Intercepción del modelo _b_0_ : 2189.865883306326, aproximadamente 2189.87 USD/M². En base a los valores
base en las columnas. 

- Coeficientes (regression.coef_), array de pendientes por columna, longitud: 64. Tiene 64 variables predictoras porque cada columna se transforma en varias columnas binarias. Indica cuanto cambia el precio por cada columna. 


##### Graficar linealidad con un gráfico de residuos:

In [ ]:
residuals_test = y_test.values - y_pred

# Predichos vs residuos
plt.figure(figsize=(8,5))
plt.scatter(y_pred, residuals_test, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Valores predichos')
plt.ylabel('Residuos (y_test - y_pred)')
plt.title('Residuos vs Valores predichos (Test set)')
plt.show()

##### Observaciones:

Los residuos se distribuyen de manera simétrica y sin patrón definido alrededor de la línea horizontal de referencia (y=0) de color rojo, lo que sugiere que el modelo presenta una buena linealidad y que los errores son aleatorios y no correlacionados.

#####  Histograma de residuos:

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(residuals_test, kde=True)
plt.title('Distribución de residuos (Test set)')
plt.show()

##### Observaciones:
- Los valores de distribución del histograma llegan a su pico alrededor del valor cero, presentan una forma aproximadamente simétrica, no se presentan sesgos marcados en valores negativos y positivos.

##### Independencia de los errores:

In [ ]:
#Para comprobar la homocedasticidad se puede examinar el gráfico de residuos generado. 
#Calculo y muestra de residuos por durbin-watson
# Calcula los residuos
residuals = y - regression.predict(X)

# Prueba de Durbin-Watson
from statsmodels.stats.stattools import durbin_watson

dw_test = durbin_watson(residuals_test)
print(f"Estadístico de Durbin-Watson: {dw_test}\n")

##### Observaciones:
- 1.9663, cercano al valor ideal 2. Los residuos del modelo son independientes entre sí, no existe evidencia de autocorrelación, y por lo tanto se cumple el supuesto de independencia de los errores. Los errores se comportan como ruido aleatorio, no como una tendencia sistemática.

In [ ]:
# 1.5 y 2.5 se utilizan como umbrales empíricos para interpretar el estadístico de Durbin-Watson.
if dw_test < 1.5:
    print("Posible autocorrelación positiva.\n")
elif dw_test > 2.5:
    print("Posible autocorrelación negativa.\n")
else:
    print("Los errores parecen ser independientes (no hay evidencia de autocorrelación).\n")


##### Grafico entrenamiento y test:

In [ ]:
plt.scatter(df['año'], df['precio_prom'], alpha=0.6)
plt.title("Relación entre año y precio promedio del m²")
plt.xlabel("Año")
plt.ylabel("Precio promedio (USD/m²)")
plt.show()

##### Observaciones:
-Se observa una distribución de precios en relación al paso de años en incremento desde 2500 USD hasta superar el umbral de los 3500 USD. Los datos propuestos por el gráfico resultan en concordancia con los esperados.

#### Validación:

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

In [ ]:
modelo_cv = LinearRegression()

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(modelo_cv, X_train, y_train, cv=kfold, scoring='r2')


##### Observaciones:
- Definimos el modelo base (regresión lineal)
- Configuramos K-Fold con 10 particiones (K = 10)
- shuffle=True para mezclar los datos antes de dividirlos
- Aplicamos la validación cruzada usando R² como métrica

In [ ]:
print(f"Resultados R² de cada fold:\n{scores}\n")
print(f"Promedio del coeficiente de determinación R²: {scores.mean():.4f}")
print(f"Desviación estándar de R²: {scores.std():.4f}\n")

##### Observaciones:
- Los datos detallados por cada fold explican un promedio aproximado de 0.80, lo cual confirma que el modelo explica un 80% de la variabilidad del precio promedio del M², lo cual es un muy buen ajuste.

- La desviación estándar baja 0.015 significa que el modelo mantiene un rendimiento estable en todas las particiones.

- Esto demuestra que el modelo está haciendo predicciones correctamente incluso con datos nuevos.

#### Error cuadrático medio negativo:

In [ ]:
mse_scores = cross_val_score(modelo_cv, X_train, y_train, cv=kfold, scoring='neg_mean_squared_error')
mse_mean = -mse_scores.mean()
print(f"Promedio del Error Cuadrático Medio (MSE) en validación cruzada: {mse_mean:.2f}\n")

#### Observaciones:
- Cuanto menor sea el MSE, más ajustado está el modelo a los datos reales.
En este caso, el error cuadrático medio de 55.000 es consistente con el obtenido antes en el test ≈54.800, lo que demuestra que el modelo generaliza bien y no sobreajusta los datos.

#### Regularización con Ridge:

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)  
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

print("R² con Ridge:", r2_score(y_test, y_pred_ridge))
print("MSE con Ridge:", mean_squared_error(y_test, y_pred_ridge))

#### Observaciones:
- El valor del R² con Ridge 0.80, sugiere un valor dentro de los parametros normalmente esperados por el modelado del trabajo. 
- El valor MSE con Ridge 54805.28, si aplicamos su raiz = 234,1. Este valor es considerado pequeño en relación a la variación de los precios almacenados en el dataframe.

#### Conclusiones:
Durante la realización del trabajo práctico se detecto una falta de datos en la columna precio_prom, se procedio a realizar la remoción de las filas las cuales tenian estos valores nulos y se manejaron los outliers reemplazandolos por las medianas de las columnas a modo de normalizar los valores distribuidos asimétricamente. Se realizó el registro de los datos originales mediante diversos gráficos y folds. Utilizando el modelo de Regresión Lineal Multiple se tomaron los datos del DataFrame y una variable dependiente. Se evaluaron los residuos para comprobar autocorrelación (igualdad en la varianza de residuos), se determinó ausencia de autocorrelación. La homocedasticidad fue evaluada visualmente mediante el gráfico de residuos vs predichos, observándose varianza aproximadamente constante. Se evaluo la predicción que formuló el modelo por validación cruzada con MSE y R², detectandose consistencia con los valores estimados mediante las evaluaciones previas. Mediante la regularización Ridge se calcularon los MSE y R², de la misma forma que la Regresion Lineal cumplió con los resultados esperados satisfactoriamente. Comparando Ridge con Regresión Lineal tenemos resultados acordes y casi similares, más especificamente; el MSE de Regresion Lineal es 54807.12 y su R²: 0.8020, en Ridge; MSE es 54805.28 y su R²: 0.8019. Se puede decir que ambas implementaciones son efectivas con valores igual de precisos y buen desempeño tratando los datos. 

Se concluye que el modelo de Regresión Lineal posee una gran adaptabilidad al Dataframe empleado en este trabajo y que los datos propuestos y los devueltos en los testeos poseen una gran concordancia.

#### Fuentes:
- Fuente del dataset:
Gobierno de la Ciudad de Buenos Aires - Portal Buenos Aires Data
(https://data.buenosaires.gob.ar/dataset/mercado-inmobiliario)

- Inteligencia artificial de apoyo: 
ChatGPT